# Tumor-cell subtyping on cytoplasmic expression (all samples)

This notebook clusters the tumor cells of each Xenium 5K tumor sample on the **cytoplasmic** expression of the
cytoplasm-enriched gene set (634 genes selected in `1_enrichment.ipynb`), annotates every cluster with a
biological program derived from its enriched genes, quantifies per-subtype characteristics (hypoxia, interferon
and other program scores, proximity to immune / T / myeloid cells, neighbourhood composition, morphology), and
summarizes the labels across samples. Each sample is subtyped **on its own** (its own resolution and clusters); the
annotated **label** comes from a program vocabulary shared by all samples, so labels are comparable, but no label is
forced to appear in every sample.

The analysis functions live in `subtype_utils.py` next to this notebook, so that `4_pooled.ipynb` (the same
procedure on all tumor cells pooled) runs exactly the same code. The sections below explain every step and name the
module function that performs it; the module docstrings carry the details.

**Inputs** (per sample, all produced upstream)
- `data/<sample>/intermediate_data/adata.h5ad`: all cells of the sample, raw whole-cell counts, cell types in `obs["cell_type_merged"]`.
- `data/<sample>/processed_data/cell_ids.npy`: ids of the tumor cells (`cell_type_merged == "Malignant cell"`).
- `output/1_exploration/<sample>/adata_cytoplasmic_embedded.h5ad`: tumor cells x gene set, built on HGCC by `2_embedding.py`
  (normalize_total on the full panel, log1p kept in `layers["lognorm"]`, depth regressed out, scaled, PCA, kNN graph, UMAP).
- `data/_utils/hallmark_pathways_filtered.gmt`, `data/_utils/all_pathways_filtered.gmt`: gene sets for the program dictionary.
- `data/<sample>/processed_data/ssgsea_hallmark_cytoplasmic.parquet`: per-cell ssGSEA scores from `2_outcome`, used as a second hypoxia definition.

**Outputs** (`output/1_exploration/<sample>/` per sample, `output/1_exploration/cross_sample/` for the summary)

| folder | content |
|---|---|
| `clustering/` | resolution sweep, cluster statistics, UMAPs, PCA loadings, tissue map by label, label composition |
| `DE/` | one marker table per cluster, marker dotplot by label |
| `annotation/` | cluster -> label table with confidence and evidence, per-cell subtype record, program-score heatmap |
| `spatial/` | one tissue map per label (alone, on immune cells, on T cells) |
| `characteristics/` | per-label summary table, per-cell table, box plots, tissue maps of scores and proximities |
| `cross_sample/` | cell-type counts, label composition and characteristics across samples, all annotations, all per-cell subtypes |

**Saved results for later stages** (keyed by `sample` + `cell_id`, so they join onto any object):
`annotation/subtype_annotation_cytoplasmic.csv` (cluster -> label, confidence, evidence),
`annotation/cell_subtypes_cytoplasmic.csv` (per cell: cluster, label, confidence, resolution) and their
concatenations `cross_sample/annotation_all_samples.csv` and `cross_sample/cell_subtypes_all_samples.csv`.

**How a sample is reviewed**
1. Run the notebook with the default entry of the sample in `SAMPLE_CONFIG` (`SAMPLES_TO_RUN` can restrict the loop).
2. Ask for a review of `clustering/resolution_sweep_cytoplasmic.csv`, `annotation/subtype_annotation_cytoplasmic.csv`,
   `clustering/cluster_stats_cytoplasmic.csv`, the `DE/` tables and `characteristics/subtype_characteristics_cytoplasmic.csv`.
3. Record the reviewed resolution and label decisions in `SAMPLE_CONFIG` (with a dated comment) and run again.

## 1. Setup

Imports, paths, the per-sample configuration (resolution, reviewed labels, tissue-map figure size) and the analysis
parameters. The parameters, colours, the program dictionary, the label vocabulary and all analysis functions come
from `subtype_utils.py` (`from subtype_utils import *`). The five output subfolders of a sample are emptied and
rewritten on every run (the sample root with the embedded objects is left alone), so stale files from earlier runs
cannot accumulate. Figures are produced without grid lines: scanpy's `set_figure_params` switches `axes.grid` on, so
the helper `set_figsize` switches it off again after every call.

In [1]:
import gc
import os
import shutil

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from mcDETECT.utils import *

import warnings
warnings.filterwarnings("ignore")
sc.settings.verbosity = 0

# parameters, colours, program dictionary, label vocabulary and the analysis functions shared with 4_pooled.ipynb
from subtype_utils import *

set_figsize((8, 8))

/Users/chenyang/miniconda3/envs/AIVC-env/lib/python3.10/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound
/Users/chenyang/miniconda3/envs/AIVC-env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/Users/chenyang/miniconda3/envs/AIVC-env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/Users/chenyang/miniconda3/envs/AIVC-env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_hdf from `annda

In [2]:
utils_path = "../../data/_utils/"
cross_path = "../../output/1_exploration/cross_sample/"
os.makedirs(cross_path, exist_ok=True)

def sample_paths(sample):
    """Input paths of a sample and its output subfolders. The sample root holds the embedded objects written by
    2_embedding.py and is never touched; the five subfolders belong to this notebook and are emptied and recreated on
    every run, so that files of labels or clusters that no longer exist (e.g. after a resolution or label change) do
    not survive next to the current outputs. Do not store anything else in them."""
    output_path = f"../../output/1_exploration/{sample}/"
    paths = {"intermediate": f"../../data/{sample}/intermediate_data/",
             "processed": f"../../data/{sample}/processed_data/",
             "output": output_path}
    for name in ["clustering", "DE", "annotation", "spatial", "characteristics"]:
        paths[name] = output_path + name + "/"
        shutil.rmtree(paths[name], ignore_errors=True)
        os.makedirs(paths[name], exist_ok=True)
    return paths

In [3]:
# Per-sample configuration.
# - res_cyto: Louvain resolution. It is sample-specific because the same value gives a different number of clusters in
#   each sample; the resolution sweep (section 4) prints a suggestion, the value here is the reviewed choice.
# - manual_labels: reviewed label decisions (cluster id -> label), applied after the automatic annotation (section 8).
#   Clusters listed here are marked "reviewed" in the annotation table. The comment records the reasoning and date.
# - fig_size: (width, height) in inches of the tissue maps, following the sample's x/y extent (BC is taller than wide,
#   y/x = 1.56; OC 0.69, CC 0.89, LC 0.68 and Prostate 0.56 are wider than tall).
SAMPLE_CONFIG = {
    # BC reviewed 2026-10-05 (13 clusters, sweep: labels saturate at 1.0). Clusters 0 (EEF1G, TSPAN13, XBP1, CCND1) and
    # 3 (PEG10, LAPTM4B, NPNT, FOXA1) are luminal identity variants without a process program -> Baseline confirmed;
    # cluster 4 (XBP1, PDIA3, AMFR, CKAP4 + HMGCR, SQLE, DHCR24) keeps the automatic "ER stress" (secretory / sterol
    # synthesis program, medium confidence); 10 is a POSTN/COL5A1 stroma border, 7 and 12 are low depth
    "Xenium_5K_BC":       {"res_cyto": 1.0, "fig_size": (7, 12), "manual_labels": {"0": "Baseline", "3": "Baseline"}},
    # OC / CC / LC / Prostate reviewed 2026-10-06. Resolution from the sweep (1.0 left complement, ER stress, hypoxia or
    # G2/M unresolved; same-label clusters are merged, so extra clusters cost nothing). Review decisions follow one
    # rule: a label is kept only if the cluster's own markers show the program; "review" Baseline clusters whose
    # markers are tissue identity genes (luminal: FOXA1, NKX3-1, XBP1; EEF1G/IMPDH2 translation) are confirmed
    # Baseline; clusters that the two criteria assign to overlapping programs of the same axis get the shared label
    # (TNFA/NFKB and p53 -> Stress response: GADD45A, CDKN1A, KLF4, NFE2L2; Glycolysis, Autophagy, mTORC1 with
    # NDRG1 / LDHA / ALDOA on top -> Hypoxia), so that the cross-sample vocabulary stays small.
    # OC (15 clusters): 1, 3, 6 Baseline (identity, no program); 7 Baseline (EEF1G, PROS1, CCND1, SOX9: the ER stress
    # score of 0.52 is not supported by ER genes); 14 Stress response (GADD45A, HSPA8, SQSTM1, TNIP1; the complement
    # score comes from CFB alone)
    "Xenium_5K_OC":       {"res_cyto": 1.5, "fig_size": (8, 6), "manual_labels": {"1": "Baseline", "3": "Baseline", "6": "Baseline",
                                                                                   "7": "Baseline", "14": "Stress response"}},
    # CC (16 clusters): 2, 6 Baseline; 3 EMT/stroma confirmed as a tumor program (CDH3, ITGA6, FAT1, MMP14; 24k cells,
    # median fibroblast distance 25.6 um, i.e. not a border artifact like cluster 9); 13 p53 -> Stress response
    "Xenium_5K_CC":       {"res_cyto": 1.2, "fig_size": (8, 7), "manual_labels": {"2": "Baseline", "6": "Baseline", "3": "EMT/stroma",
                                                                                   "13": "Stress response"}},
    # LC (17 clusters; deep sample, ~700 cytoplasmic transcripts per cell, so marker lists are long and hallmark
    # overlaps fire easily): 0 Autophagy -> Hypoxia (NDRG1, ALDOA, NQO1, BLVRA; hypoxia score 0.36) and 9 Glycolysis
    # -> Hypoxia (LDHA, ALDOA; hypoxia score 0.53); 4 ER stress (PDIA3, PDIA4, HYOU1, SDF2L1, DNAJA1; deepest and
    # largest cells); 8, 13 Baseline; 14 Oxidative stress -> Baseline (GABRB3, TCF4, DACH1, BEX2, PEG10: a
    # neural-like identity variant, not oxidative stress); 15 EMT/stroma -> Baseline (148 cells, PLAT, IGFBP2, no
    # collagen genes)
    "Xenium_5K_LC":       {"res_cyto": 1.5, "fig_size": (8, 6), "manual_labels": {"0": "Hypoxia", "4": "ER stress", "8": "Baseline",
                                                                                   "9": "Hypoxia", "13": "Baseline", "14": "Baseline",
                                                                                   "15": "Baseline"}},
    # Prostate (17 clusters): 0, 3, 5, 11 Baseline (luminal: NKX3-1, FOXA1, XBP1, SPON2); 7 ER stress (PDIA3, PDIA4,
    # HYOU1, NUCB2 + DHCR24, EBP, like the BC ER stress cluster); 2, 9 TNFA/NFKB -> Stress response (ZFAND5, GADD45A,
    # TRIB1, NFE2L2, CDKN1A); 8 mTORC1 -> Hypoxia (NDRG1 top, hypoxia score 0.83); 13 Glycolysis -> Baseline (LDHA is
    # the only glycolysis gene among SOX9, XBP1, IGFBP2 luminal markers)
    "Xenium_5K_Prostate": {"res_cyto": 1.5, "fig_size": (8, 5), "manual_labels": {"0": "Baseline", "3": "Baseline", "5": "Baseline",
                                                                                   "11": "Baseline", "7": "ER stress", "2": "Stress response",
                                                                                   "9": "Stress response", "8": "Hypoxia", "13": "Baseline"}},
}
SAMPLES_TO_RUN = list(SAMPLE_CONFIG)   # restrict to a subset while reviewing, e.g. ["Xenium_5K_OC"]

# Analysis parameters (RES_GRID, LOGFC_THR, PCT_THR, P_OVERLAP, Z_SCORE, MIN_TARGET_COUNTS, BORDER_DIST, RADIUS,
# MIN_PROGRAM_GENES) and the reference cell groups of the spatial context (spatial_refs) are defined in subtype_utils.py,
# shared with the pooled analysis
print(f"RES_GRID={RES_GRID}, LOGFC_THR={LOGFC_THR}, PCT_THR={PCT_THR}, P_OVERLAP={P_OVERLAP}, Z_SCORE={Z_SCORE}, "
      f"MIN_TARGET_COUNTS={MIN_TARGET_COUNTS}, BORDER_DIST={BORDER_DIST}, RADIUS={RADIUS}, MIN_PROGRAM_GENES={MIN_PROGRAM_GENES}")
for name, cell_types in spatial_refs.items():
    print(f"{name:12s} {cell_types}")

RES_GRID=[0.5, 0.8, 1.0, 1.2, 1.5], LOGFC_THR=0.5, PCT_THR=0.1, P_OVERLAP=0.001, Z_SCORE=0.5, MIN_TARGET_COUNTS=20, BORDER_DIST=25, RADIUS=30, MIN_PROGRAM_GENES=5
immune       ['CD4+ T cell', 'CD8+ T cell', 'T cell', 'B cell', 'Dendritic cell', 'Myeloid cell', 'Mast cell']
tcell        ['CD4+ T cell', 'CD8+ T cell', 'T cell']
cd8_tcell    ['CD8+ T cell']
myeloid      ['Myeloid cell', 'Dendritic cell']
bcell        ['B cell']
fibroblast   ['Fibroblast (CAF)']
endothelial  ['Endothelial cell']


## 2. Cell types across samples

The spatial references above are names of `obs["cell_type_merged"]`. This cell reads that column of every sample
(h5py, without loading the matrices), tabulates the cell types, and checks that the names used in `spatial_refs`
exist. A reference group that is empty in a sample gets NaN distances / proximities and zero neighbourhood fractions
there; names outside the known vocabulary (`known_types` in the module) are printed as a warning so that a renamed
type is noticed before the loop.

In [4]:
cell_type_counts = {}
for s in SAMPLE_CONFIG:
    with h5py.File(f"../../data/{s}/intermediate_data/adata.h5ad", "r") as f:
        group = f["obs"]["cell_type_merged"]
        categories_s = group["categories"][:].astype(str)
        codes = group["codes"][:]
    cell_type_counts[s] = pd.Series(categories_s[codes]).value_counts()
cell_type_counts = pd.DataFrame(cell_type_counts).fillna(0).astype(int)
cell_type_counts = cell_type_counts.loc[cell_type_counts.sum(axis=1).sort_values(ascending=False).index]
cell_type_counts.to_csv(cross_path + "cell_type_counts.csv")

unexpected = sorted(set(cell_type_counts.index) - known_types)
if unexpected:
    print(f"WARNING: cell types outside the known vocabulary: {unexpected}")
for name, cell_types in spatial_refs.items():
    assert any(ct in cell_type_counts.index for ct in cell_types), f"reference group {name}: none of {cell_types} exists in any sample"
for s in SAMPLE_CONFIG:
    present = set(cell_type_counts.index[cell_type_counts[s] > 0])
    empty = [name for name, cell_types in spatial_refs.items() if not (set(cell_types) & present)]
    partial = {name: sorted(set(cell_types) & present) for name, cell_types in spatial_refs.items()
               if (set(cell_types) & present) and not set(cell_types) <= present}
    print(f"{s}: {cell_type_counts[s].sum()} cells, {cell_type_counts.loc['Malignant cell', s]} tumor cells; "
          f"empty reference groups: {empty if empty else 'none'}; partial: {partial if partial else 'none'}")
cell_type_counts

Xenium_5K_BC: 402871 cells, 102180 tumor cells; empty reference groups: none; partial: none
Xenium_5K_OC: 327607 cells, 160250 tumor cells; empty reference groups: none; partial: none
Xenium_5K_CC: 588646 cells, 221355 tumor cells; empty reference groups: none; partial: none
Xenium_5K_LC: 278328 cells, 44624 tumor cells; empty reference groups: none; partial: {'immune': ['B cell', 'CD4+ T cell', 'CD8+ T cell', 'Dendritic cell', 'Mast cell', 'Myeloid cell'], 'tcell': ['CD4+ T cell', 'CD8+ T cell']}
Xenium_5K_Prostate: 193000 cells, 95429 tumor cells; empty reference groups: ['bcell']; partial: {'immune': ['CD8+ T cell', 'Mast cell', 'Myeloid cell'], 'tcell': ['CD8+ T cell'], 'myeloid': ['Myeloid cell']}


,Xenium_5K_BC,Xenium_5K_OC,Xenium_5K_CC,Xenium_5K_LC,Xenium_5K_Prostate
Malignant cell,102180,160250,221355,44624,95429
Fibroblast (CAF),52305,58591,173145,35821,21085
Smooth muscle cell,430,39224,38859,16446,26699
Endothelial cell,19374,16403,25084,38033,9761
Myeloid cell,32969,15818,22719,22631,7547
CD4+ T cell,47481,1882,25857,23362,0
Epithelial cell (non-malignant),34310,8237,11041,21561,0
B cell,40304,847,12092,17710,0
CD8+ T cell,29625,2694,14480,16214,6631
Dendritic cell,22827,1681,15733,21480,0


## 3. Gene programs and per-cell program scores

The clusters are annotated against a fixed dictionary of biological programs so that every sample receives the same
vocabulary (`load_programs_raw`, `PROGRAM_NAMES`). Most programs are Hallmark or GO-BP sets from the project GMT files.
Five programs are curated lists (`CURATED_PROGRAMS`), because the filtered GMTs do not isolate them: the hallmark
interferon-gamma and complement sets share CFB, HIF1A, CDKN1A, CTSL and LGMN with other programs, and no hallmark set
carries the HSPA8 / HSPB8 / SQSTM1 heat-shock program, the lysosomal program (CTSL, LGMN, LIPA, GRN) or sterol
synthesis (HMGCR, SQLE, DHCR24). A sixth curated list, macrophage-specific genes (ITGB2, SIRPA, MSR1, CD68, CYBB, ...),
detects clusters that carry spill-over transcripts of neighbouring myeloid cells ("Myeloid border", an artifact class
like the stroma border). The hallmark interferon / complement sets and the estrogen-response sets (tissue identity,
not a process) are kept as scores only (`score_only`) and never used as a label.

Every program is restricted to the embedding gene set (`build_programs`); programs with fewer than
`MIN_PROGRAM_GENES` genes left are dropped. Each cell is then scored for each program (`score_programs`) with
`sc.tl.score_genes` (mean of the program genes minus the mean of expression-matched control genes) on the
log-normalized cytoplasmic expression, and the scores are z-scored over cells. The scores drive the resolution sweep,
the annotation and the subtype characteristics. The label vocabulary (`LABEL_ORDER`) and its colours
(`LABEL_COLORS`) are derived from the full dictionary, so they are identical in every sample and in the pooled analysis.

In [5]:
programs_raw = load_programs_raw(utils_path)
print(f"{len(programs_raw)} programs: " + ", ".join(f"{name} ({len(genes)})" for name, genes in programs_raw.items()))
print(f"label vocabulary ({len(LABEL_ORDER)}): {LABEL_ORDER}")

23 programs: S-phase (111), G2/M (116), Interferon (51), Complement / acute phase (31), Stress response (42), Lysosome (38), Cholesterol synthesis (29), Hypoxia (96), Glycolysis (74), EMT/stroma (100), ER stress (109), Oxidative stress (220), Autophagy (108), TNFA/NFKB (110), Apoptosis (103), p53 (88), mTORC1 (92), MYC targets (79), Myeloid border (47), Estrogen early (96), Estrogen late (93), Interferon gamma (hallmark) (106), Complement (hallmark) (103)
label vocabulary (22): ['S-phase', 'G2/M', 'Interferon', 'Complement / acute phase', 'Stress response', 'Lysosome', 'Cholesterol synthesis', 'Hypoxia', 'Glycolysis', 'EMT/stroma', 'ER stress', 'Oxidative stress', 'Autophagy', 'TNFA/NFKB', 'Apoptosis', 'p53', 'mTORC1', 'MYC targets', 'Baseline', 'Stroma border', 'Myeloid border', 'Low depth']


## 4. Load a sample

`adata_tumor` holds the tumor cells with raw whole-cell counts and all cell metadata (coordinates, areas, cell
types); `adata_tumor_cyto` holds the same cells on the cytoplasmic expression of the gene set, with the depth-corrected
embedding and the kNN graph from `2_embedding.py`. The two objects must have identical cell order.

In [6]:
def load_sample(paths):
    adata = sc.read_h5ad(paths["intermediate"] + "adata.h5ad")
    cell_ids = np.load(paths["processed"] + "cell_ids.npy", allow_pickle=True)
    assert adata.shape[0] > cell_ids.shape[0], "cell_ids should be a subset of adata.obs_names"
    adata_tumor = adata[adata.obs["cell_id"].isin(cell_ids)].copy()

    adata_tumor_cyto = sc.read_h5ad(paths["output"] + "adata_cytoplasmic_embedded.h5ad")
    assert adata_tumor.shape[0] == adata_tumor_cyto.shape[0], "adata_tumor and adata_tumor_cyto should have the same number of cells"
    assert (adata_tumor.obs["cell_id"].values == adata_tumor_cyto.obs["cell_id"].values).all(), "cell order differs between the two objects"
    print(f"{adata.shape[0]} cells in the sample, {adata_tumor.shape[0]} tumor cells, {adata_tumor_cyto.shape[1]} genes in the gene set")
    return adata, adata_tumor, adata_tumor_cyto

## 5. Resolution sweep and clustering (`resolution_sweep`, `cluster`)

Louvain clustering runs on the kNN graph saved by `2_embedding.py` (30 PCs, 30 neighbours). The resolution is
sample-specific: the sweep reports, for each value in `RES_GRID`, the number of clusters, the smallest cluster, and
the number of distinct programs that the score criterion alone (mean z > `Z_SCORE`) resolves. The suggested
resolution is the smallest one at which that number stops increasing, i.e. the coarsest clustering that still
separates all detectable programs. The clustering itself uses `cfg["res_cyto"]`, the reviewed value, so a review can
override the suggestion. The cluster ids are sample-specific; the labels assigned in section 8 are the unit of
comparison across samples, and several clusters may share one label.

## 6. Cluster diagnostics (`diagnostics` = `spatial_context` + `cluster_statistics`)

Two checks that a cluster is a biological substate rather than an artifact:

- **Depth.** The cytoplasmic vectors are sparse (median ~130 transcripts per cell on the panel, ~40 on the gene set),
  and without correction PC1 is sequencing depth and the clusters are depth strata. `2_embedding.py` regresses depth
  out; the Spearman correlations of the first PCs with log depth printed here should all be below 0.3 in absolute
  value, and the median depth of the clusters should be within ~2x of each other. A cluster with a median of fewer
  than `MIN_TARGET_COUNTS` target-gene transcripts is a low-depth residue of cells that are nearly all zeros (flagged).
- **Spatial context.** For every tumor cell the distance (um) to the nearest cell of each reference group is
  computed, and from it a **proximity** = -log10(distance + 1), so that a higher value means closer (the log
  compresses the long tail of far cells). Distances stay in the tables because the stroma-border rule is defined in
  um; proximities are what the characteristics and the figures use. The composition of the neighbourhood within
  `RADIUS` um (fraction of immune cells, of T cells, and the number of cells = crowding) is computed as well. A
  cluster close to fibroblasts with collagen markers, or one enriched for macrophage genes, sits at a tissue border
  and most likely reflects segmentation spill-over from the neighbouring cells rather than a tumor substate.

## 7. Differential expression (`differential_expression`)

Wilcoxon test of each cluster against all other tumor cells on the log-normalized cytoplasmic expression of the gene
set (`layers["lognorm"]`; `X` holds the regressed and scaled values and is not suitable). With ~1e5 cells almost every
gene is statistically significant, so a marker must also pass an effect-size threshold (log2 fold change >
`LOGFC_THR`) and an expression threshold (expressed in more than `PCT_THR` of the cluster's cells). One table per
cluster is written to `DE/`, sorted by the Wilcoxon score.

## 8. Annotation (`annotate`, `save_cell_subtypes`)

Each cluster is labelled automatically from its enriched genes with two independent criteria:

1. **Marker overlap.** Hypergeometric test of the cluster's marker list against each label program, with the
   embedding gene set as the universe. The program with the smallest p labels the cluster if p < `P_OVERLAP`.
2. **Program score.** Mean z of each program score over the cluster's cells (section 3). If no program passes the
   overlap test, the top-scoring program labels the cluster if its mean z > `Z_SCORE`.

A cluster that passes neither criterion is **Baseline** (tissue identity without a process program). Two overrides
follow: **Low depth** if the cluster's median target-gene count is below `MIN_TARGET_COUNTS`; **Stroma border** if the
label is EMT/stroma and the cluster's median distance to fibroblasts is below `BORDER_DIST` um (collagen / POSTN
transcripts of adjacent fibroblasts spilling into the tumor cells' cytoplasmic compartment). The curated **Myeloid
border** program plays the same role for macrophage transcripts (ITGB2, SIRPA, MSR1, CD68, ...): it is a label of its
own, listed with the artifact classes, and a cluster that receives it is a myeloid-contact zone rather than a
substate. A genuine EMT program far from fibroblasts keeps the EMT/stroma label.

The `confidence` column says how sure the rule is: *high* when both criteria name the same program (or a Baseline /
Low depth cluster has fewer than 10 markers), *medium* when only one criterion fires, *review* when the two criteria
disagree or a Baseline / Low depth cluster has 10 or more markers. The review step (see the top of the notebook)
resolves the *review* clusters; its decisions go into `cfg["manual_labels"]`, are applied last and marked *reviewed*.
Clusters with the same label are merged into `subtype_label`; the cluster id stays in `subtype_louvain_cyto`. The
per-cell record (`annotation/cell_subtypes_cytoplasmic.csv`: sample, cell_id, cluster, label, confidence, resolution)
is the compact result that later stages join on.

## 9. Subtype characteristics (`characteristics` = `cell_characteristics` + `summarize_characteristics`)

Per-cell quantities summarized per label (median and inter-quartile range) in
`characteristics/subtype_characteristics_cytoplasmic.csv`; the per-cell values go to
`characteristics/cell_annotations_cytoplasmic.csv`, which the cross-sample comparison and the pooled analysis read.

- **Hypoxia**, three definitions: the Hallmark hypoxia program score on cytoplasmic expression (`score_Hypoxia`,
  section 3); the ssGSEA score of the same set on cytoplasmic expression from `2_outcome` (`hypoxia_ssgsea_cyto`); and
  the stress score of `2_outcome/5_score.ipynb`, the 2nd-highest expression among eight hypoxia transcription factors
  on whole-cell log-normalized counts (`hypoxia_score_tf`). The last one is the model's stress definition; at Xenium
  depth it is zero for most cells, so it is kept in the tables for comparability but not plotted.
- **Program scores** for interferon, complement, proliferation, stress response and lysosome (section 3) and the
  ssGSEA interferon-gamma score.
- **Proximity** to immune, T, CD8+ T, myeloid, B, fibroblast and endothelial cells (section 6; higher = closer),
  with the raw distances kept in the per-cell table.
- **Neighbourhood**: immune and T-cell fraction and the number of cells within `RADIUS` um.
- **Depth and morphology**: cytoplasmic depth, target-gene counts, cell area, nucleus-to-cell area ratio.

## 10. Figures (`make_figures`)

| figure | folder | content |
|---|---|---|
| `umap_clustering.png`, `pca_loadings.png` | `clustering/` | UMAP by cluster id; genes with the largest loadings on PC1-5 (sanity check: program genes, not depth) |
| `umap_subtype_label.png`, `scatter_subtype_label.png` | `clustering/` | UMAP and tissue map coloured by label |
| `subtype_composition.png` | `clustering/` | percentage of tumor cells per label |
| `heatmap_program_scores.png` | `annotation/` | mean program-score z per label: the evidence behind each label |
| `dotplot_subtype_markers.png` | `DE/` | top 5 markers per label, mean expression scaled per gene and fraction of expressing cells |
| `scatter_<label>.png`, `_immune_cell.png`, `_tcell.png` | `spatial/` | one label on the other tumor cells, on the immune cells, on the T cells |
| `boxplot_characteristics.png` | `characteristics/` | per-label box plots of scores, proximities, neighbourhood and morphology |
| `scatter_score_*.png`, `scatter_proximity_*.png` | `characteristics/` | tissue maps of per-cell values (clipped at the 1st and 99th percentile) |

The tissue maps use the sample's global coordinates at the figure size of `SAMPLE_CONFIG`.

## 11. Run the pipeline per sample

`run_sample` executes sections 4-10 for one sample with its own configuration and returns the tables that the
cross-sample summary needs; the AnnData objects are released before the next sample. Each sample is clustered and
annotated independently; the only things shared are the gene set, the program vocabulary and the parameters.

In [7]:
def run_sample(sample):
    cfg = SAMPLE_CONFIG[sample]
    paths = sample_paths(sample)
    print(f"\n{'=' * 30} {sample} {'=' * 30}")

    adata, adata_tumor, adata_tumor_cyto = load_sample(paths)
    programs, label_programs = build_programs(set(adata_tumor_cyto.var_names), programs_raw)
    scores_z = score_programs(adata_tumor_cyto, adata_tumor, programs)

    print(f"\n--- {sample}: resolution sweep and clustering")
    sweep = resolution_sweep(adata_tumor_cyto, scores_z, label_programs, cfg, paths)
    cluster(adata_tumor_cyto, adata_tumor, cfg["res_cyto"])

    print(f"\n--- {sample}: diagnostics")
    cluster_stats = diagnostics(adata, adata_tumor, adata_tumor_cyto, paths)

    print(f"\n--- {sample}: differential expression")
    de_dict = differential_expression(adata_tumor_cyto, paths)

    print(f"\n--- {sample}: annotation")
    annotation, categories = annotate(adata_tumor_cyto, adata_tumor, de_dict, cluster_stats, scores_z, programs, label_programs, cfg, paths)
    cell_subtypes = save_cell_subtypes(adata_tumor, annotation, sample, cfg["res_cyto"], paths["annotation"] + "cell_subtypes_cytoplasmic.csv")

    print(f"\n--- {sample}: characteristics")
    summary = characteristics(adata_tumor, paths)

    make_figures(adata.obs, adata_tumor, adata_tumor_cyto, de_dict, annotation, categories, scores_z, sample, paths, cfg["fig_size"])

    result = {"sweep": sweep, "cluster_stats": cluster_stats, "annotation": annotation, "characteristics": summary,
              "cell_subtypes": cell_subtypes, "n_tumor_cells": int(adata_tumor.shape[0])}
    del adata, adata_tumor, adata_tumor_cyto, scores_z, de_dict
    gc.collect()
    return result

results = {}
for sample in SAMPLES_TO_RUN:
    results[sample] = run_sample(sample)


============================== Xenium_5K_BC ==============================


402871 cells in the sample, 102180 tumor cells, 634 genes in the gene set
S-phase                       47 genes: ANP32E, BIRC5, CBX5, CCNB2, CCNE1, CDC20, CDKN1A, CDKN2A, CDKN3, CHEK1, ...
G2/M                          40 genes: BIRC5, BUB1, CCNA2, CCNB2, CCND1, CDC20, CDKN3, CENPF, CHEK1, CKS2, ...
Interferon                    15 genes: IFIT1, IFIT2, IFIT3, IFI35, IFIH1, STAT1, IFITM1, RSAD2, EIF2AK2, GBP1, ...
Complement / acute phase       7 genes: SERPINA3, CFB, C4A, C4B, UBD, CFH, CD59
Stress response               10 genes: HSPA8, HSPB8, DNAJA1, BAG3, SQSTM1, KLF4, CDKN1A, GADD45A, ZFAND5, SERPINH1
Lysosome                      10 genes: CTSL, LGMN, LIPA, GRN, CREG1, GLB1, GLA, MAN2B1, LAPTM4B, GNS
Cholesterol synthesis          6 genes: HMGCR, SQLE, DHCR24, FDFT1, ACLY, EBP
Hypoxia                       26 genes: ALDOA, ANXA2, CCN2, CDKN1A, CDKN1C, COL5A1, CXCR4, FBP1, HK2, LDHA, ...
Glycolysis                    22 genes: ALDOA, CD44, COL5A1, CXCR4, FUT8, GFPT1, GOT1, HK2, HM


--- Xenium_5K_BC: resolution sweep and clustering


 resolution  n_clusters  min_cluster_size  n_program_labels                                                                                   programs
        0.5           6                33                 3                                                            EMT/stroma, Interferon, S-phase
        0.8           9                33                 5                                 EMT/stroma, Interferon, Lysosome, S-phase, Stress response
        1.0          13                33                 7 Complement / acute phase, EMT/stroma, G2/M, Interferon, Lysosome, S-phase, Stress response
        1.2          14                33                 7 Complement / acute phase, EMT/stroma, G2/M, Interferon, Lysosome, S-phase, Stress response
        1.5          18                33                 7 Complement / acute phase, EMT/stroma, G2/M, Interferon, Lysosome, S-phase, Stress response
Suggested resolution: 1.0 (config uses 1.0)


Number of clusters (cytoplasmic, resolution 1.0): 13

--- Xenium_5K_BC: diagnostics


Spearman(PC1, log10 cytoplasmic depth) = -0.20
Spearman(PC2, log10 cytoplasmic depth) = +0.05
Spearman(PC3, log10 cytoplasmic depth) = +0.00
Spearman(PC4, log10 cytoplasmic depth) = +0.03
Spearman(PC5, log10 cytoplasmic depth) = -0.02
                      n_cells  median_cyto_depth  median_cyto_target_counts  median_cell_area  median_dist_immune  median_dist_fibroblast  low_depth
subtype_louvain_cyto                                                                                                                                
0                       15011              127.0                       39.0             105.6                27.4                    42.8      False
1                       12013              125.0                       33.0             103.8                28.1                    38.8      False
2                       11444              116.0                       35.0             100.6                23.3                    34.7      False
3                   

... (42 outputs left out)

Cluster 0: 30 marker genes; top: SPON2, IGFBP2, PDIA3, PDIA4, CDH1, NUCB2, LBH, HYOU1
Cluster 1: 11 marker genes; top: EEF1G, HSPA8, NKX3-1, SORD, KIF5B, EIF3A, IMPDH2, SESN3
Cluster 2: 59 marker genes; top: ZFAND5, ANXA2, GADD45A, NFE2L2, DKK3, SFRP1, CCN2, APP
Cluster 3: 20 marker genes; top: EEF1G, XBP1, IGFBP2, PDIA3, NKX3-1, CDH1, FMOD, BCAM
Cluster 4: 0 marker genes; top: 
Cluster 5: 22 marker genes; top: FOXA1, NKX3-1, IL1R1, XBP1, SORD, DAP, ARF6, EEF1G
Cluster 6: 82 marker genes; top: EEF1G, IMPDH2, PCBD1, PHB2, PABPC4, HSPA8, NOA1, IRAK1
Cluster 7: 71 marker genes; top: PRXL2A, DHCR24, IGFBP2, PDIA3, PDIA4, NKX3-1, CD47, NUCB2
Cluster 8: 28 marker genes; top: NDRG1, SORD, TGFB3, PAK1IP1, NBL1, DHCR24, ANXA2, CCDC86
Cluster 9: 20 marker genes; top: TRIB1, XBP1, CDKN1A, LBH, EFNB2, LDHA, FOXA1, PTP4A1
Cluster 10: 57 marker genes; top: SERPINA3, C4A, CFB, BACE2, XBP1, SOX9, C4B, ANXA2
Cluster 11: 13 marker genes; top: EEF1G, TUBB, IMPDH2, CD44, PHB2, MRPL45, FSTL1, NCOA4
Cluster


--- Xenium_5K_Prostate: characteristics


Spearman(score_Hypoxia, hypoxia_ssgsea_cyto) = +0.55
Spearman(score_Hypoxia, hypoxia_score_tf) = +0.08
Spearman(hypoxia_ssgsea_cyto, hypoxia_score_tf) = +0.21


                          n_cells  score_Hypoxia  hypoxia_ssgsea_cyto  hypoxia_score_tf  score_Interferon  interferon_ssgsea_cyto  score_Complement / acute phase  score_S-phase  score_G2/M  score_Stress response  score_Lysosome  proximity_immune  proximity_tcell  proximity_cd8_tcell  proximity_myeloid  proximity_bcell  proximity_fibroblast  proximity_endothelial  frac_immune_30um  frac_tcell_30um  n_cells_30um  cyto_depth  cyto_target_counts  cell_area  nucleus_cell_area_ratio
subtype_label                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    
S-phase                      2229   

## 12. Cross-sample summary

Labels are compared across the samples that were run (`output/1_exploration/cross_sample/`):

- `label_composition.csv` / `.png`: percentage of tumor cells per label and sample (0 where a label is absent) and the
  number of samples in which each label occurs.
- `label_characteristics.csv` / `.png`: per-sample, per-label medians of the characteristics; the figure shows, for a
  few key metrics, one marker per sample at each label.
- `annotation_all_samples.csv`: every cluster of every sample with its label, confidence and evidence (the review record).
- `cell_subtypes_all_samples.csv`: the per-cell subtype records of all samples in one table (sample, cell_id,
  cluster, label, confidence, resolution), the input of `4_pooled.ipynb` and of later stages.

In [8]:
samples_run = list(results)

# composition: labels x samples (% of tumor cells)
composition = pd.DataFrame({s: results[s]["annotation"].groupby("label")["n_cells"].sum() / results[s]["n_tumor_cells"] * 100
                            for s in samples_run}).fillna(0)
composition = composition.loc[[label for label in LABEL_ORDER if label in composition.index]]
composition["n_samples"] = (composition[samples_run] > 0).sum(axis=1)
composition.round(2).to_csv(cross_path + "label_composition.csv")

fig, ax = plt.subplots(figsize=(0.9 * len(composition) + 2, 4.5))
width = 0.8 / len(samples_run)
x = np.arange(len(composition))
for i, s in enumerate(samples_run):
    ax.bar(x + (i - (len(samples_run) - 1) / 2) * width, composition[s].values, width, label=s.replace("Xenium_5K_", ""),
           color=plt.get_cmap("tab10")(i))
ax.grid(False)
ax.set_xticks(x)
ax.set_xticklabels(composition.index, rotation=60, ha="right")
ax.set_ylabel("% of tumor cells")
ax.legend(frameon=False)
plt.savefig(cross_path + "label_composition.png", dpi=300, bbox_inches="tight")
plt.close()

# characteristics: long table of per-label medians per sample
label_chars = pd.concat([results[s]["characteristics"].assign(sample=s).rename_axis("label").reset_index() for s in samples_run])
label_chars = label_chars[["sample", "label"] + [c for c in label_chars.columns if c not in ("sample", "label")]]
label_chars.round(4).to_csv(cross_path + "label_characteristics.csv", index=False)

metrics = {"score_Hypoxia": "Hypoxia score (hallmark)", "score_Interferon": "Interferon score",
           "proximity_immune": "Proximity to immune cell", "proximity_tcell": "Proximity to T cell",
           f"frac_immune_{RADIUS}um": f"Immune fraction within {RADIUS} um"}
fig, axes = plt.subplots(1, len(metrics), figsize=(5.2 * len(metrics), 4.5))
for ax, (col, title) in zip(axes, metrics.items()):
    for i, s in enumerate(samples_run):
        d = label_chars[label_chars["sample"] == s].set_index("label").reindex(composition.index)
        ax.plot(np.arange(len(composition)), d[col].values, "o", color=plt.get_cmap("tab10")(i), label=s.replace("Xenium_5K_", ""))
    ax.grid(False)
    ax.set_xticks(np.arange(len(composition)))
    ax.set_xticklabels(composition.index, rotation=60, ha="right")
    ax.set_title(title)
axes[0].legend(frameon=False)
plt.tight_layout()
plt.savefig(cross_path + "label_characteristics.png", dpi=200, bbox_inches="tight")
plt.close()

# all annotations (review record) and all per-cell subtype records
annotation_all = pd.concat([results[s]["annotation"].assign(sample=s).reset_index() for s in samples_run])
annotation_all = annotation_all[["sample"] + [c for c in annotation_all.columns if c != "sample"]]
annotation_all.to_csv(cross_path + "annotation_all_samples.csv", index=False)
cell_subtypes_all = pd.concat([results[s]["cell_subtypes"] for s in samples_run])
cell_subtypes_all.to_csv(cross_path + "cell_subtypes_all_samples.csv", index=False)

print(f"labels per sample: " + ", ".join(f"{s.replace('Xenium_5K_', '')}: {int((composition[s] > 0).sum())}" for s in samples_run))
print("clusters still marked 'review': " + (", ".join(f"{r.sample} {r.cluster}" for r in annotation_all[annotation_all['confidence'] == 'review'].itertuples()) or "none"))
print(f"{len(cell_subtypes_all)} cells in cell_subtypes_all_samples.csv")
composition.round(1)

labels per sample: BC: 10, OC: 8, CC: 11, LC: 10, Prostate: 9
clusters still marked 'review': none
623838 cells in cell_subtypes_all_samples.csv


,Xenium_5K_BC,Xenium_5K_OC,Xenium_5K_CC,Xenium_5K_LC,Xenium_5K_Prostate,n_samples
label,,,,,,
S-phase,9.5,13.8,10.4,6.6,2.3,5
G2/M,7.9,10.9,10.0,0.0,0.0,3
Interferon,5.3,3.1,6.5,12.8,0.0,4
Complement / acute phase,6.2,0.0,0.0,12.2,4.5,3
Stress response,11.2,1.7,14.9,0.0,15.7,4
Lysosome,2.8,0.0,0.0,9.8,0.0,2
Cholesterol synthesis,0.0,0.0,0.0,0.0,1.5,1
Hypoxia,0.0,14.4,13.2,20.4,5.7,4
EMT/stroma,0.0,0.0,11.0,0.0,0.0,1
